# Import Libraries

In [1]:
import pandas as pd
import great_expectations as gx
from great_expectations.data_context import FileDataContext

/opt/miniconda3/envs/h8_env/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/opt/miniconda3/envs/h8_env/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


# Load Data

In [2]:
context = gx.get_context(context_root_dir='./gx/')

In [3]:
df = pd.read_csv('dags/P2M3_soraya_intan_data_clean.csv')

# Delete datasource if already exists
try:
    context.delete_datasource('pandas_datasource')
except:
    pass

# Re-create datasource
datasource = context.sources.add_pandas(name='pandas_datasource')
data_asset = datasource.add_dataframe_asset(name='clean_data_asset')

# Build batch request
batch_request = data_asset.build_batch_request(dataframe=df)

# Create an Expectation Suite

In [7]:
# Creat an expectation suite
expectation_suite_name = 'p2m3_expectation_suite'
context.add_or_update_expectation_suite(expectation_suite_name)

# Create a validator using above expectation suite
validator = context.get_validator(
    batch_request = batch_request,
    expectation_suite_name = expectation_suite_name
)

# Check the validator
validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,id,warehouse_block,mode_of_shipment,customer_care_calls,customer_rating,cost_of_the_product,prior_purchases,product_importance,gender,discount_offered,weight_in_gms,reachedontime_yn
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


## Expectation

In [8]:
# 1. Unique
validator.expect_column_values_to_be_unique('id')

result = validator.validate()
print("Validation Success:", result.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Validation Success: True


In [9]:
# 2. Between
validator.expect_column_values_to_be_between('customer_rating', min_value=1, max_value=5)

result = validator.validate()
print("Validation Success:", result.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Validation Success: True


In [10]:
# 3. In set
validator.expect_column_values_to_be_in_set('reachedontime_yn', [0, 1])

result = validator.validate()
print("Validation Success:", result.success)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/18 [00:00<?, ?it/s]

Validation Success: True


In [11]:
# 4. Type list
validator.expect_column_values_to_be_in_type_list('customer_care_calls', ['int64'])

result = validator.validate()
print("Validation Success:", result.success)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/18 [00:00<?, ?it/s]

Validation Success: True


In [12]:
# 5. Schema Match
validator.expect_table_columns_to_match_set(list(df.columns))

result = validator.validate()
print("Validation Success:", result.success)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/18 [00:00<?, ?it/s]

Validation Success: True


In [13]:
# 6. Row Count
validator.expect_table_row_count_to_equal(10999)

result = validator.validate()
print("Validation Success:", result.success)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/18 [00:00<?, ?it/s]

Validation Success: True


In [14]:
# 7. Column Values Length
validator.expect_column_value_lengths_to_be_between(
    'warehouse_block',
    min_value=1,
    max_value=1
)

result = validator.validate()
print("Validation Success:", result.success)

Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/24 [00:00<?, ?it/s]

Validation Success: True


In [15]:
# Save expectation suite
validator.save_expectation_suite(discard_failed_expectations=False)

1. Unique (id): Tidak ada duplikasi, data bersih secara identitas

2. Between (customer_rating): Semua nilai di range 1–5, sesuai domain bisnis

3. In set (reachedontime_yn): Hanya 0 dan 1, tidak ada noise

4. Type (customer_care_calls): Semua integer

5. Schema match: Struktur dataset konsisten

6. Row count: Jumlah baris data sesuai dengan jumlah data yang digunakan, yaitu 10.999 baris.

7. Value length (`warehouse_block`): Nilai pada kolom warehouse memiliki panjang karakter yang konsisten.

Secara keseluruhan, data sudah bersih, valid

## Checkpoint

In [16]:
# Create a checkpoint

checkpoint = context.add_or_update_checkpoint(
    name="checkpoint_1",
    validations=[
        {
            "batch_request": batch_request,
            "expectation_suite_name": expectation_suite_name,
        }
    ],
)

In [17]:
# Run a checkpoint

checkpoint_result = checkpoint.run(
    batch_request=batch_request
)

Calculating Metrics:   0%|          | 0/32 [00:00<?, ?it/s]

Checkpoint berhasil dijalankan dengan total 32 metrik validasi yang seluruhnya terpenuhi (100%). Hal ini menunjukkan bahwa seluruh expectation yang telah didefinisikan berhasil diterapkan secara konsisten pada dataset.